In [4]:
import os
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/ajcode007/Data-science-project.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "ajcode007"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "aa5912f227a0080f80f1af2751a38a00cefb52af"

In [5]:
%pwd

'c:\\Users\\AJAY\\Documents\\ML Projects\\Data-science-project\\research'

In [6]:
os.chdir("../")
%pwd

'c:\\Users\\AJAY\\Documents\\ML Projects\\Data-science-project'

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [8]:
from src.Datascienceproject.constants import *
from src.Datascienceproject.utils.common import read_yaml,create_directories,save_json

In [9]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation(self) -> ModelEvaluationConfig:
        config =  self.config.model_evaluation
        params =  self.params.ElasticNet
        schema =  self.schema.TARGET_COLUMNS

        create_directories([config.root_dir])

        model_evaluation_config =  ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            metric_file_name=config.metric_file_name,
            all_params=params,
            target_column=schema.name,
            mlflow_uri= "https://dagshub.com/ajcode007/Data-science-project.mlflow"
        )
        return model_evaluation_config

In [10]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

c:\Users\AJAY\Documents\ML Projects\Data-science-project\.venv310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self,actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]


        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
            
            # Saving metrics as local
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)


            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")


In [12]:
import inspect
print(ModelEvaluationConfig)                         # shows the class object & where it’s defined
print(ModelEvaluationConfig.__dataclass_fields__.keys())  # dataclass fields
print(inspect.signature(ModelEvaluationConfig))      # constructor signature

<class '__main__.ModelEvaluationConfig'>
dict_keys(['root_dir', 'test_data_path', 'model_path', 'all_params', 'metric_file_name', 'target_column', 'mlflow_uri'])
(root_dir: pathlib.Path, test_data_path: pathlib.Path, model_path: pathlib.Path, all_params: dict, metric_file_name: pathlib.Path, target_column: str, mlflow_uri: str) -> None


In [13]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()

except Exception as e:
    raise e

[2026-07-31 22:19:30,237:INFO:common:yaml file:config\config.yaml loaded successfully]
[2026-07-31 22:19:30,241:INFO:common:yaml file:params.yaml loaded successfully]
[2026-07-31 22:19:30,246:INFO:common:yaml file:schema.yaml loaded successfully]
[2026-07-31 22:19:30,250:INFO:common:created directory at : artifacts]
[2026-07-31 22:19:30,252:INFO:common:created directory at : artifacts/model_evaluation]
[2026-07-31 22:19:32,291:INFO:common:json file saved at: artifacts\model_evaluation\metrics.json]


2026/07/31 22:19:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'ElasticnetModel'.
2026/07/31 22:20:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 1
Created version '1' of model 'ElasticnetModel'.


🏃 View run fun-shark-264 at: https://dagshub.com/ajcode007/Data-science-project.mlflow/#/experiments/0/runs/e3557558a2994b3c97368ac80be8c5fa
🧪 View experiment at: https://dagshub.com/ajcode007/Data-science-project.mlflow/#/experiments/0
